<a href="https://colab.research.google.com/github/MarcelDiaz/HW2-BigData/blob/main/HW2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Práctica HW2 Big Data. \
Marcel, Pau, Ane

#**Imports**

In [5]:
from pyspark.sql.functions import col, sum

#**Introducing the Data: Yellow Taxis from January and February 2026**

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Crear la sesión de Spark
spark = SparkSession.builder \
    .appName('NYC_Taxi_Analysis') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

spark

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

os.listdir('/content/drive/MyDrive/')

['Activitats',
 'diagrama_usuaris.drawio',
 'Diagrama sin título',
 'yellow_tripdata_2026-01.parquet',
 'yellow_tripdata_2026-02.parquet']

In [ ]:
# Rutas en Google Drive
path_jan = '/content/drive/MyDrive/yellow_tripdata_2026-01.parquet'
path_feb = '/content/drive/MyDrive/yellow_tripdata_2026-02.parquet'

# Leer datos
df_jan = spark.read.parquet(path_jan)
df_feb = spark.read.parquet(path_feb)

# Unificar
df = df_jan.union(df_feb)

# Inspección
df.printSchema()
print(f'Total de registros: {df.count()}')

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

Total de registros: 7124755


##Data Preprocessing

In [6]:
# Conteo inicial para documentar
total_inicial = df.count()
print(f"Total de registros iniciales: {total_inicial}")

# Calcular la cantidad de valores nulos por cada columna
nulos_por_columna = df.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in df.columns
])

print("Cantidad de valores nulos por columna:")
nulos_por_columna.show(vertical=True)

Total de registros iniciales: 7124755
Cantidad de valores nulos por columna:
-RECORD 0------------------------
 VendorID              | 0       
 tpep_pickup_datetime  | 0       
 tpep_dropoff_datetime | 0       
 passenger_count       | 2111375 
 trip_distance         | 0       
 RatecodeID            | 2111375 
 store_and_fwd_flag    | 2111375 
 PULocationID          | 0       
 DOLocationID          | 0       
 payment_type          | 0       
 fare_amount           | 0       
 extra                 | 0       
 mta_tax               | 0       
 tip_amount            | 0       
 tolls_amount          | 0       
 improvement_surcharge | 0       
 total_amount          | 0       
 congestion_surcharge  | 2111375 
 Airport_fee           | 2111375 
 cbd_congestion_fee    | 0       



In [8]:
# Opción A (Recomendada): Eliminar registros sin datos de tarifa, distancia o pasajeros
columnas_importantes = ["passenger_count", "trip_distance", "fare_amount"]
df_limpio = df.dropna(subset=columnas_importantes)

# Opción B (Alternativa): Imputar (rellenar) los pasajeros nulos con 1 (el mínimo lógico)
# df_limpio = df_limpio.fillna({"passenger_count": 1})

# Conteo final para ver cuántos registros perdimos
total_sin_nulos = df_limpio.count()
registros_eliminados = total_inicial - total_sin_nulos

print(f"Total de registros tras limpiar nulos: {total_sin_nulos}")
print(f"Registros eliminados por falta de datos: {registros_eliminados} ({(registros_eliminados/total_inicial)*100:.2f}%)")

Total de registros tras limpiar nulos: 5013380
Registros eliminados por falta de datos: 2111375 (29.63%)


In [9]:
# Guardamos el conteo antes de este paso para el informe
total_antes_filtros = df_limpio.count()

# 1. Filtro de distancia: Un viaje debe tener una distancia mayor a 0
df_limpio = df_limpio.filter(col("trip_distance") > 0)

# 2. Filtro de tarifas: Eliminamos tarifas en cero o negativas (suelen ser cancelaciones o errores)
df_limpio = df_limpio.filter((col("fare_amount") > 0) & (col("total_amount") > 0))

# 3. Filtro de tiempo (Duración): La fecha/hora de bajada debe ser estrictamente mayor a la de subida.
# Esto previene duraciones cero o negativas antes de que crees la variable en el Punto 3.
df_limpio = df_limpio.filter(col("tpep_dropoff_datetime") > col("tpep_pickup_datetime"))

# Conteo posterior y cálculo de eliminados
total_despues_filtros = df_limpio.count()
registros_filtrados = total_antes_filtros - total_despues_filtros

print(f"Registros eliminados por ceros/negativos: {registros_filtrados}")
print(f"Total de registros válidos hasta ahora: {total_despues_filtros}")

Registros eliminados por ceros/negativos: 209369
Total de registros válidos hasta ahora: 4804011
